In [23]:
!pip -q install transformers sentencepiece sacremoses torch pandas


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import torch

In [25]:
import pandas as pd

glossary_url = "https://raw.githubusercontent.com/tarikvuk1/RestaurantMenuTranslator/main/data/processed/traditional_dish_glossary_10.csv"

glossary = pd.read_csv(glossary_url)

print("Number of dishes:", len(glossary))
display(glossary.head())

Number of dishes: 32


,bosnian_name,english_description,typical_ingredients,meat_content,potential_allergens,notes
0,Francuska supa od luka,French onion soup topped with melted cheese.,"Onions, beef or vegetable broth, white wine, c...",Possible beef broth,"Milk, gluten, possible sulfites",French dish; ingredients depend on the broth.
1,Pohovana paprika sa sirom,Breaded pepper stuffed with cheese and fried u...,"Peppers, cheese, breadcrumbs, flour, eggs, oil",NaN,"Milk, gluten, eggs",Common Balkan appetizer; cheese type may vary.
2,Wok s piletinom,Stir-fried chicken with vegetables and sauce.,"Chicken, mixed vegetables, soy sauce, oil, spices",Chicken,"Soy, wheat, possible sesame",Sauce ingredients vary; check for soy and wheat.
3,Bosanska plata,"Traditional Bosnian platter of cured meat, sau...","Beef prosciutto, Bosnian sudžuka, Travnik cheese",Beef,"Milk, possible gluten",Served as a cold appetizer; meat selection var...
4,Uštipci sa kajmakom,Small pieces of fried dough served with rich B...,"Wheat flour, water or milk, yeast or baking po...",NaN,"Gluten, milk, possible eggs",Traditional breakfast or appetizer.


In [26]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

print("Loading translation model...")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    src_lang="bos_Latn"
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

print("Translation model loaded successfully")


Loading translation model...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Translation model loaded successfully


In [27]:
import torch

def translate_menu(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    with torch.no_grad():
        translated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(
                "eng_Latn"
            ),
            max_new_tokens=128
        )

    return tokenizer.batch_decode(
        translated_tokens,
        skip_special_tokens=True
    )[0]

In [28]:
test_dishes = [
    "Ćevapi sa lukom",
    "Begova čorba",
    "Klepe sa pavlakom",
    "Pileća supa",
    "Burek sa mesom"
]

for dish in test_dishes:
    translation = translate_menu(dish)

    print("Bosnian:", dish)
    print("English:", translation)
    print("-" * 40)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bosnian: Ćevapi sa lukom
English: The bow with the bow
----------------------------------------


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bosnian: Begova čorba
English: Beg's soup
----------------------------------------


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bosnian: Klepe sa pavlakom
English: Strap with a stick
----------------------------------------


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bosnian: Pileća supa
English: Chicken soup
----------------------------------------
Bosnian: Burek sa mesom
English: Burk with meat
----------------------------------------


In [29]:
import pandas as pd

baseline_results = []

for dish in test_dishes:
    translation = translate_menu(dish)

    baseline_results.append({
        "bosnian": dish,
        "baseline_translation": translation
    })

baseline_df = pd.DataFrame(baseline_results)

display(baseline_df)

baseline_df.to_csv(
    "baseline_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Baseline results saved successfully")


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

,bosnian,baseline_translation
0,Ćevapi sa lukom,The bow with the bow
1,Begova čorba,Beg's soup
2,Klepe sa pavlakom,Strap with a stick
3,Pileća supa,Chicken soup
4,Burek sa mesom,Burk with meat


Baseline results saved successfully


In [30]:
import os

print("Current folder:", os.getcwd())
print("CSV file exists:", os.path.exists("baseline_results.csv"))
print("Full file path:", os.path.abspath("baseline_results.csv"))


Current folder: c:\Users\Lana\OneDrive\Desktop\RestaurantMenuTranslator
CSV file exists: True
Full file path: c:\Users\Lana\OneDrive\Desktop\RestaurantMenuTranslator\baseline_results.csv


In [31]:
print("Glossary columns:")
print(glossary.columns.tolist())

print("\nFirst 3 dishes:")
display(glossary.head(3))


Glossary columns:
['bosnian_name', 'english_description', 'typical_ingredients', 'meat_content', 'potential_allergens', 'notes']

First 3 dishes:


,bosnian_name,english_description,typical_ingredients,meat_content,potential_allergens,notes
0,Francuska supa od luka,French onion soup topped with melted cheese.,"Onions, beef or vegetable broth, white wine, c...",Possible beef broth,"Milk, gluten, possible sulfites",French dish; ingredients depend on the broth.
1,Pohovana paprika sa sirom,Breaded pepper stuffed with cheese and fried u...,"Peppers, cheese, breadcrumbs, flour, eggs, oil",NaN,"Milk, gluten, eggs",Common Balkan appetizer; cheese type may vary.
2,Wok s piletinom,Stir-fried chicken with vegetables and sauce.,"Chicken, mixed vegetables, soy sauce, oil, spices",Chicken,"Soy, wheat, possible sesame",Sauce ingredients vary; check for soy and wheat.


In [32]:
import unicodedata
import re

def normalize_text(text):
    text = str(text).lower().strip()

    replacements = {
        "č": "c",
        "ć": "c",
        "š": "s",
        "ž": "z",
        "đ": "dj"
    }

    for original, replacement in replacements.items():
        text = text.replace(original, replacement)

    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char for char in text
        if not unicodedata.combining(char)
    )

    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [33]:
test_names = [
    "Ćevapi",
    "Cevapi",
    "Begova čorba",
    "Begova corba",
    "KLEPE",
    "Šopska salata"
]

for name in test_names:
    print(name, "->", normalize_text(name))


Ćevapi -> cevapi
Cevapi -> cevapi
Begova čorba -> begova corba
Begova corba -> begova corba
KLEPE -> klepe
Šopska salata -> sopska salata


In [34]:
from difflib import SequenceMatcher

def find_dish(text, glossary, threshold=0.75):
    normalized_input = normalize_text(text)

    best_match = None
    best_score = 0

    for _, row in glossary.iterrows():
        dish_name = row["bosnian_name"]
        normalized_dish = normalize_text(dish_name)

        score = SequenceMatcher(
            None,
            normalized_input,
            normalized_dish
        ).ratio()

        if score > best_score:
            best_score = score
            best_match = row

    if best_score >= threshold:
        return {
            "dish": best_match["bosnian_name"],
            "description": best_match["english_description"],
            "ingredients": best_match["typical_ingredients"],
            "meat": best_match["meat_content"],
            "allergens": best_match["potential_allergens"],
            "score": round(best_score, 2)
        }

    return None


In [35]:
test_inputs = [
    "Cevpapi",
    "Begova corba",
    "Kleepe",
    "Burekk",
    "Pizza"
]

for text in test_inputs:
    result = find_dish(text, glossary)

    print("\nInput:", text)

    if result:
        print("Recognized:", result["dish"])
        print("Description:", result["description"])
        print("Similarity:", result["score"])
    else:
        print("No matching Bosnian dish found")



Input: Cevpapi
No matching Bosnian dish found

Input: Begova corba
Recognized: Begova čorba
Description: Traditional creamy Bosnian soup made with chicken, vegetables and okra.
Similarity: 1.0

Input: Kleepe
Recognized: Klepe
Description: Traditional Bosnian dumplings filled with minced meat and served with garlic yogurt sauce.
Similarity: 0.91

Input: Burekk
No matching Bosnian dish found

Input: Pizza
No matching Bosnian dish found


In [36]:
def translate_with_glossary(text):

    baseline = translate_menu(text)

    match = find_dish(text, glossary)

    if match:
        return {
            "original": text,
            "baseline": baseline,
            "improved": match["description"],
            "dish": match["dish"],
            "ingredients": match["ingredients"],
            "meat": match["meat"],
            "allergens": match["allergens"]
        }

    return {
        "original": text,
        "baseline": baseline,
        "improved": baseline,
        "dish": None,
        "ingredients": None,
        "meat": None,
        "allergens": None
    }


In [37]:
test_inputs = [
    "Ćevapi",
    "Begova čorba",
    "Klepe",
    "Burek",
    "Pileća supa"
]

for text in test_inputs:
    result = translate_with_glossary(text)

    print("\nBosnian:", result["original"])
    print("Baseline:", result["baseline"])
    print("Glossary:", result["improved"])
    print("-" * 40)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Bosnian: Ćevapi
Baseline: ♪ The cough ♪
Glossary: ♪ The cough ♪
----------------------------------------


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Bosnian: Begova čorba
Baseline: Beg's soup
Glossary: Traditional creamy Bosnian soup made with chicken, vegetables and okra.
----------------------------------------


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Bosnian: Klepe
Baseline: The clippers
Glossary: Traditional Bosnian dumplings filled with minced meat and served with garlic yogurt sauce.
----------------------------------------


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Bosnian: Burek
Baseline: Burek
Glossary: Burek
----------------------------------------

Bosnian: Pileća supa
Baseline: Chicken soup
Glossary: Chicken soup
----------------------------------------


In [38]:
from difflib import SequenceMatcher

def find_dish_in_phrase(text, glossary, threshold=0.80):
    normalized_input = normalize_text(text)
    input_words = normalized_input.split()

    best_match = None
    best_score = 0
    best_position = None

    for _, row in glossary.iterrows():
        dish_name = row["bosnian_name"]
        dish_words = normalize_text(dish_name).split()

        n = len(dish_words)

        for i in range(len(input_words) - n + 1):
            phrase = " ".join(input_words[i:i+n])
            dish = " ".join(dish_words)

            score = SequenceMatcher(
                None, phrase, dish
            ).ratio()

            if score > best_score:
                best_score = score
                best_match = row
                best_position = (i, i+n)

    if best_match is not None and best_score >= threshold:
        start, end = best_position

        remaining_words = (
            input_words[:start] + input_words[end:]
        )

        return {
            "dish": best_match["bosnian_name"],
            "description": best_match["english_description"],
            "ingredients": best_match["typical_ingredients"],
            "meat": best_match["meat_content"],
            "allergens": best_match["potential_allergens"],
            "extra_text": " ".join(remaining_words),
            "score": round(best_score, 2)
        }

    return None


In [39]:
test_phrases = [
    "Ćevapi sa lukom",
    "Klepe sa pavlakom",
    "Burek sa mesom",
    "Begova corba",
    "Cevpapi sa lukom",
    "Pizza sa sirom"
]

for phrase in test_phrases:
    result = find_dish_in_phrase(phrase, glossary)

    print("\nInput:", phrase)

    if result:
        print("Recognized dish:", result["dish"])
        print("Description:", result["description"])
        print("Extra text:", result["extra_text"])
        print("Similarity:", result["score"])
    else:
        print("No traditional dish recognized")

    print("-" * 40)



Input: Ćevapi sa lukom
Recognized dish: Ćevapi sa lukom
Description: Traditional Bosnian grilled minced meat sausages served with chopped raw onions, often accompanied by somun bread.
Extra text: 
Similarity: 1.0
----------------------------------------

Input: Klepe sa pavlakom
Recognized dish: Klepe
Description: Traditional Bosnian dumplings filled with minced meat and served with garlic yogurt sauce.
Extra text: sa pavlakom
Similarity: 1.0
----------------------------------------

Input: Burek sa mesom
Recognized dish: Burek sa mesom
Description: Traditional Bosnian savory pastry made with thin layers of dough filled with seasoned minced meat.
Extra text: 
Similarity: 1.0
----------------------------------------

Input: Begova corba
Recognized dish: Begova čorba
Description: Traditional creamy Bosnian soup made with chicken, vegetables and okra.
Extra text: 
Similarity: 1.0
----------------------------------------

Input: Cevpapi sa lukom
Recognized dish: Ćevapi sa lukom
Descriptio

In [40]:
menu_phrases = {
    "sa lukom": "with onions",
    "sa pavlakom": "with sour cream",
    "sa mesom": "with meat",
    "sa sirom": "with cheese",
    "sa kajmakom": "with kajmak (creamy dairy spread)",
    "sa krompirom": "with potatoes",
    "bez luka": "without onions",
    "bez mesa": "without meat"
}

def translate_extra_text(text):
    normalized = normalize_text(text)

    if not normalized:
        return ""

    if normalized in menu_phrases:
        return menu_phrases[normalized]

    # Fallback to pretrained translation model
    return translate_menu(text)


In [41]:
test_extras = [
    "sa lukom",
    "sa pavlakom",
    "sa mesom",
    "bez luka",
    ""
]

for extra in test_extras:
    print(
        repr(extra),
        "->",
        translate_extra_text(extra)
    )


'sa lukom' -> with onions
'sa pavlakom' -> with sour cream
'sa mesom' -> with meat
'bez luka' -> without onions
'' -> 


In [42]:
def final_translate_menu(text):
    baseline = translate_menu(text)
    match = find_dish_in_phrase(text, glossary)

    if match:
        extra_translation = translate_extra_text(
            match["extra_text"]
        )

        return {
            "original": text,
            "baseline": baseline,
            "dish": match["dish"],
            "description": match["description"],
            "extra_translation": extra_translation,
            "ingredients": match["ingredients"],
            "meat": match["meat"],
            "allergens": match["allergens"],
            "similarity": match["score"],
            "method": "glossary"
        }

    return {
        "original": text,
        "baseline": baseline,
        "dish": None,
        "description": baseline,
        "extra_translation": "",
        "ingredients": None,
        "meat": None,
        "allergens": None,
        "similarity": None,
        "method": "baseline"
    }


In [43]:
test_menu_items = [
    "Ćevapi sa lukom",
    "Klepe sa pavlakom",
    "Burek sa mesom",
    "Begova čorba",
    "Pileća supa",
    "Pizza sa sirom"
]

for item in test_menu_items:
    result = final_translate_menu(item)

    print("\nOriginal:", result["original"])
    print("Baseline:", result["baseline"])
    print("Description:", result["description"])
    print("Extra:", result["extra_translation"])
    print("Method:", result["method"])
    print("-" * 50)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Original: Ćevapi sa lukom
Baseline: The bow with the bow
Description: Traditional Bosnian grilled minced meat sausages served with chopped raw onions, often accompanied by somun bread.
Extra: 
Method: glossary
--------------------------------------------------


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Original: Klepe sa pavlakom
Baseline: Strap with a stick
Description: Traditional Bosnian dumplings filled with minced meat and served with garlic yogurt sauce.
Extra: with sour cream
Method: glossary
--------------------------------------------------


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Original: Burek sa mesom
Baseline: Burk with meat
Description: Traditional Bosnian savory pastry made with thin layers of dough filled with seasoned minced meat.
Extra: 
Method: glossary
--------------------------------------------------


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Original: Begova čorba
Baseline: Beg's soup
Description: Traditional creamy Bosnian soup made with chicken, vegetables and okra.
Extra: 
Method: glossary
--------------------------------------------------

Original: Pileća supa
Baseline: Chicken soup
Description: Chicken soup
Extra: 
Method: baseline
--------------------------------------------------


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Original: Pizza sa sirom
Baseline: Pizza with cheese
Description: Pizza with cheese
Extra: 
Method: baseline
--------------------------------------------------


In [44]:
import pandas as pd

results = []

for item in test_menu_items:
    result = final_translate_menu(item)
    results.append(result)

results_df = pd.DataFrame(results)

display(results_df)

results_df.to_csv(
    "glossary_translation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Results saved successfully")


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

,original,baseline,dish,description,extra_translation,ingredients,meat,allergens,similarity,method
0,Ćevapi sa lukom,The bow with the bow,Ćevapi sa lukom,Traditional Bosnian grilled minced meat sausag...,,"Minced beef, onions, salt, pepper, somun bread",Usually beef,Possible gluten,1.0,glossary
1,Klepe sa pavlakom,Strap with a stick,Klepe,Traditional Bosnian dumplings filled with minc...,with sour cream,"Wheat flour, minced beef, onions, garlic, yogu...",Usually beef,"Gluten, milk, possible eggs",1.0,glossary
2,Burek sa mesom,Burk with meat,Burek sa mesom,Traditional Bosnian savory pastry made with th...,,"Wheat flour, minced beef, onions, oil, salt, p...",Usually beef,"Gluten, possible eggs",1.0,glossary
3,Begova čorba,Beg's soup,Begova čorba,Traditional creamy Bosnian soup made with chic...,,"Chicken, okra, carrots, celery, flour, cream",Chicken,"Possible gluten, milk, celery",1.0,glossary
4,Pileća supa,Chicken soup,NaN,Chicken soup,,NaN,NaN,NaN,NaN,baseline
5,Pizza sa sirom,Pizza with cheese,NaN,Pizza with cheese,,NaN,NaN,NaN,NaN,baseline


Results saved successfully
